
# 09 · Recommendation audit

What do real people actually get, and which input causes what? A workbench to look at outcomes side by side before changing the app, so nothing in the ranking is a black box.

What is validated and what isn't:

| Part | Status |
|---|---|
| **Reach**: where people from a job actually move (LightGBM + CPS history) | validated: hit@10 0.62 on held-out 2022-25 moves (notebook 04) |
| Text placement (description / CV) | measured in notebook 07: helps on pseudo-CVs, weak on short real text |
| Skill answers (15 questions) | nudge the title's profile by 25%; survey: ~1.2 points noise |
| Interest fit (topics, quiz), goal multipliers, experience multiplier, the "best fit" blend | **design choices, not validated** |

Inputs: the 17 survey respondents (title, description, 15 skill answers) and Tim's own saved profile.


In [1]:

import sys, json
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
from api.engine import Engine, Profile

engine = Engine()
pd.set_option('display.max_colwidth', 40)



## Inputs

Survey respondents become profiles: the job title is Tim's assigned occupation (as if they picked it from the dropdown), plus their description and 15 skill answers.


In [2]:

survey = pd.read_csv(ROOT / 'data/raw/survey/responses.csv')
key_of = {item['name']: item['key'] for item in engine.meta['items']}
item_cols = {c: key_of[c.replace('Knowledge of ', '').replace('Skill: ', '')] for c in survey.columns[3:]}

TRUE_JOB = [110, 1107, 1020, 1030, 1600, 5860, 1860, 1020, 1610, 1860, 2010, 2000, 2100, 2100, 2010, 1860, 1030]

people = []
for i, row in survey.iterrows():
    description = row.iloc[2] if isinstance(row.iloc[2], str) and len(row.iloc[2].strip()) > 10 else None
    people.append({
        'who': f"#{i + 1} {row.iloc[1].strip()}",
        'job': TRUE_JOB[i],
        'answers': {item_cols[c]: float(row[c]) for c in item_cols if pd.notna(row[c])},
        'description': description,
    })

tim = Profile(**json.loads((ROOT / 'data/raw/survey/tim-profile.json').read_text())['profile'])          # before the CV
tim_cv = Profile(**json.loads((ROOT / 'data/raw/survey/tim-profile-cv.json').read_text())['profile'])   # after the CV
len(people), engine.census.at[tim.job, 'title']


(17, 'Computer and information systems managers')


## Looking inside one recommendation

`components(profile)` returns every candidate stop with the numbers behind its rank: reach (share of real moves and its rank), skill match, interest fit (`appeal_pct`), and the final blended `best`. Sort by any column to see what drives the order.


In [3]:

COLUMNS = ['title', 'reach_label', 'reach_rank', 'reach_share', 'skill_match', 'appeal_pct', 'median_wage', 'bls_ai_exposure', 'best']


def components(profile, n=15, sort='best'):
    rec = engine.recommend(profile, sort=sort, limit=600)
    frame = pd.DataFrame(rec['results']).set_index('census')
    return frame[[c for c in COLUMNS if c in frame]].head(n)


components(tim)


,title,reach_label,reach_rank,reach_share,skill_match,appeal_pct,median_wage,bls_ai_exposure,best
census,,,,,,,,,
2310,Elementary and middle school teachers,Occasional move,32.0,0.0046,0.2443,0.8743,64100.5111,High,0.9870
2320,Secondary school teachers,Rare move,53.0,0.0013,0.2593,0.8752,71507.7922,High,0.8731
2630,Designers,Rare move,68.0,0.0005,0.1302,0.8462,58546.5358,High,0.7100
3060,Physicians and surgeons,Occasional move,29.0,0.0052,0.1690,0.7002,277303.1575,High,0.7018
420,Social and community service managers,Occasional move,14.0,0.0118,0.3086,0.5111,80390.0000,Very high,0.6970
2300,Preschool and kindergarten teachers,Rare move,96.0,0.0003,0.0885,0.8656,42069.7926,Moderate,0.6744
2340,Other teachers and instructors,Rare move,74.0,0.0004,0.1031,0.7988,44378.2159,High,0.6479
2100,Lawyers,Occasional move,12.0,0.0129,0.3291,0.5309,159670.0000,Very high,0.6255
2330,Special education teachers,Rare move,93.0,0.0003,0.2464,0.7640,71449.4768,High,0.6124



## Approaches

Each takes a profile and returns the top stops. Add your own.

- **current**: the app today (blended best fit)
- **natural next steps**: reach only (the validated part)
- **toward what you love**: interest fit only, among stops that are at least a rare-but-real move (reach rank ≤ 150)


In [4]:

def current(profile, n=6):
    return components(profile, n, 'best')['title'].tolist()


def natural_next_steps(profile, n=6):
    return components(profile, n, 'reachable')['title'].tolist()


def toward_what_you_love(profile, n=6, max_reach_rank=150):
    frame = components(profile, 600, 'appealing')
    if frame['appeal_pct'].isna().all():
        return []
    if frame['reach_rank'].notna().any():
        frame = frame[frame['reach_rank'] <= max_reach_rank]
    return frame['title'].head(n).tolist()


def top_moves_by_interest(profile, n=6, pool=20):
    """Real moves decide what's on the list (top `pool` by reach, the validated zone);
    interests only decide the order within it. Without interests: plain reach order."""
    frame = components(profile, 600, 'reachable').nsmallest(pool, 'reach_rank')
    if frame['appeal_pct'].notna().any():
        frame = frame.sort_values('appeal_pct', ascending=False)
    return frame['title'].head(n).tolist()


def core_moves_by_interest(profile, n=6, coverage=0.8, min_skill=0.0):
    """The pool is the moves that together make up `coverage` of all real moves from here (not a fixed 20),
    minus stops whose skills don't match at all (skill_match <= min_skill: physicians from tech, etc.).
    Interests only order the pool."""
    frame = components(profile, 600, 'reachable').sort_values('reach_rank')
    pool = frame[frame['reach_share'].cumsum().shift(fill_value=0) < coverage * frame['reach_share'].sum()]
    pool = pool[pool['skill_match'] > min_skill]
    if pool['appeal_pct'].notna().any():
        pool = pool.sort_values('appeal_pct', ascending=False)
    return pool['title'].head(n).tolist()


def moves_with_interest_nudge(profile, n=6, low=0.5):
    """Real moves decide; interests only nudge: score = move share x (low + interest fit).
    An interest match can lift a stop at most x1.5 and sink it to x0.5 (low=0.5), so it moves a few places,
    but a rare move can't jump a common one."""
    frame = components(profile, 600, 'reachable')
    nudge = (low + frame['appeal_pct']).fillna(1.0)
    return frame.assign(score=frame['reach_share'] * nudge).sort_values('score', ascending=False)['title'].head(n).tolist()


APPROACHES = {'current': current, 'natural next steps': natural_next_steps, 'toward what you love': toward_what_you_love,
              'top 20 moves, by interest': top_moves_by_interest,
              'core moves (80%, skills fit), by interest': core_moves_by_interest,
              'moves, interest nudge (app next)': moves_with_interest_nudge}

pd.DataFrame({name: fn(tim) for name, fn in APPROACHES.items()})


,current,natural next steps,toward what you love,"top 20 moves, by interest","core moves (80%, skills fit), by interest","moves, interest nudge (app next)"
0,Elementary and middle school teachers,Marketing and sales managers,Physical therapists,"Software developers, applications an...","Software developers, applications an...",Marketing and sales managers
1,Secondary school teachers,Financial managers,"Architects, except naval",Computer network architects,Education administrators,Financial managers
2,Designers,"Compliance, project & other managers",Secondary school teachers,Education administrators,Public relations and fundraising man...,"Compliance, project & other managers"
3,Physicians and surgeons,Administrative services managers,Elementary and middle school teachers,Architectural and engineering managers,Lawyers,Administrative services managers
4,Social and community service managers,Public relations and fundraising man...,Preschool and kindergarten teachers,Public relations and fundraising man...,Social and community service managers,Public relations and fundraising man...
5,Preschool and kindergarten teachers,Education administrators,Clergy,Lawyers,Advertising and promotions managers,Education administrators


## Tim before and after the CV, every approach

Before: title = computer and information systems managers, 15 answers. After: title = computer and information research scientists + software developers, CV activities and tools, 6 years.

In [5]:
pd.concat({
    'before CV': pd.DataFrame({name: fn(tim) for name, fn in APPROACHES.items()}),
    'after CV': pd.DataFrame({name: fn(tim_cv) for name, fn in APPROACHES.items()}),
})

current  \
before CV 0  Elementary and middle school teachers   
          1              Secondary school teachers   
          2                              Designers   
          3                Physicians and surgeons   
          4  Social and community service managers   
          5    Preschool and kindergarten teachers   
after CV  0  Elementary and middle school teachers   
          1                              Designers   
          2              Secondary school teachers   
          3               Architects, except naval   
          4            Artists and related workers   
          5    Preschool and kindergarten teachers   

                                  natural next steps  \
before CV 0             Marketing and sales managers   
          1                       Financial managers   
          2     Compliance, project & other managers   
          3         Administrative services managers   
          4  Public relations and fundraising man...   
          5                 Education administrators   
after CV  0                Computer systems analysts   
          1                     Computer programmers   
          2             Computer support specialists   
          3                           Web developers   
          4            Information security analysts   
          5             Operations research analysts   

                              toward what you love  \
before CV 0                    Physical therapists   
          1               Architects, except naval   
          2              Secondary school teachers   
          3  Elementary and middle school teachers   
          4    Preschool and kindergarten teachers   
          5                                 Clergy   
after CV  0               Architects, except naval   
          1                   Biomedical engineers   
          2                               Drafters   
          3                    Materials engineers   
          4              Secondary school teachers   
          5  Elementary and middle school teachers   

                           top 20 moves, by interest  \
before CV 0  Software developers, applications an...   
          1             Computer network architects    
          2                 Education administrators   
          3   Architectural and engineering managers   
          4  Public relations and fundraising man...   
          5                                  Lawyers   
after CV  0                 Architects, except naval   
          1                                Designers   
          2       Robotics, energy & other engineers   
          3  Remote sensing & other physical scie...   
          4                           Web developers   
          5  Environmental scientists and geoscie...   

            core moves (80%, skills fit), by interest  \
before CV 0  Software developers, applications an...    
          1                 Education administrators    
          2  Public relations and fundraising man...    
          3                                  Lawyers    
          4    Social and community service managers    
          5      Advertising and promotions managers    
after CV  0                                Designers    
          1       Robotics, energy & other engineers    
          2  Remote sensing & other physical scie...    
          3                           Web developers    
          4     Electrical and electronics engineers    
          5  Environmental scientists and geoscie...    

                    moves, interest nudge (app next)  
before CV 0             Marketing and sales managers  
          1                       Financial managers  
          2     Compliance, project & other managers  
          3         Administrative services managers  
          4  Public relations and fundraising man...  
          5                 Education administrators  
after CV  0                Computer systems analysts


## Tim's profile: which input does what?

The same approach with one input switched off at a time.


In [6]:

ablations = {
    'as entered': tim,
    'no goals': replace(tim, goals=[]),
    'no interests (topics + quiz)': replace(tim, chips=[], quiz={}),
    'no skill answers': replace(tim, answers={}),
    'experience 7 years': replace(tim, experience_years=7),
    'title only': Profile(job=tim.job),
}
pd.DataFrame({name: current(p) for name, p in ablations.items()}).T


,0,1,2,3,4,5
as entered,Elementary and middle school teachers,Secondary school teachers,Designers,Physicians and surgeons,Social and community service managers,Preschool and kindergarten teachers
no goals,"Software developers, applications an...",Elementary and middle school teachers,Postsecondary teachers,Physicians and surgeons,Advertising and promotions managers,"Compliance, project & other managers"
no interests (topics + quiz),Marketing and sales managers,"Compliance, project & other managers",Medical and health services managers,Social and community service managers,Construction managers,Advertising and promotions managers
no skill answers,Elementary and middle school teachers,Secondary school teachers,Physicians and surgeons,Designers,Social and community service managers,Preschool and kindergarten teachers
experience 7 years,Elementary and middle school teachers,Secondary school teachers,Architectural and engineering managers,Education administrators,Designers,Physicians and surgeons
title only,Marketing and sales managers,Financial managers,"Compliance, project & other managers",Administrative services managers,Public relations and fundraising man...,Advertising and promotions managers



## The 17 respondents: what does each input add?

Same person, growing inputs. `answers only` is the questionnaire without a title: can the 15 questions place someone on their own?


In [7]:

def sweep(person, n=4):
    base = Profile(job=person['job'])
    rows = {
        'title only': natural_next_steps(base, n),
        '+ skill answers': natural_next_steps(replace(base, answers=person['answers']), n),
    }
    if person['description']:
        rows['+ description'] = natural_next_steps(replace(base, answers=person['answers'], description=person['description']), n)
    rows['answers only (no title)'] = natural_next_steps(Profile(answers=person['answers']), n)
    return rows


for person in people:
    print(person['who'], '->', engine.census.at[person['job'], 'title'])
    for name, tops in sweep(person).items():
        print(f'   {name:24s}', ' | '.join(t[:28] for t in tops))


#1 Engineering manager -> Computer and information systems managers
   title only               Marketing and sales managers | Financial managers | Compliance, project & other  | Administrative services mana
   + skill answers          Financial managers | Marketing and sales managers | Compliance, project & other  | Administrative services mana
   + description            Financial managers | Marketing and sales managers | Compliance, project & other  | Administrative services mana
   answers only (no title)  Nursing, psychiatric, and ho | Registered nurses | Medical assistants | Physicians and surgeons
#2 Product designer -> IT project managers, QA testers, UI designers & more tech


   title only               Software developers, applica | Computer support specialists | Computer systems analysts | Network and computer systems
   + skill answers          Software developers, applica | Computer support specialists | Computer systems analysts | Network and computer systems
   + description            Computer support specialists | Computer systems analysts | Network and computer systems | Computer programmers
   answers only (no title)  Artists and related workers | Producers and directors | Janitors and building cleane | Writers and authors
#3 Software engineer -> Software developers, applications and systems software
   title only               IT project managers, QA test | Computer systems analysts | Computer support specialists | Computer programmers
   + skill answers          IT project managers, QA test | Computer systems analysts | Computer support specialists | Computer programmers
   + description            Computer systems analysts | Computer support sp

   title only               Software developers, applica | IT project managers, QA test | Computer support specialists | Computer systems analysts
   + skill answers          Software developers, applica | IT project managers, QA test | Computer support specialists | Computer systems analysts
   answers only (no title)  Assemblers & fabricators | Inspectors, testers, sorters | Welding, soldering, and braz | First-line supervisors of pr
#5 Research Scientist -> Agricultural and food scientists
   title only               Medical scientists | Remote sensing & other physi | Psychologists | Biological scientists
   + skill answers          Medical scientists | Remote sensing & other physi | Psychologists | Biological scientists
   answers only (no title)  Remote sensing & other physi | Quality control, environment | Registered nurses | Medical scientists
#6 Office Coordinator -> Office clerks, general


   title only               Secretaries and administrati | Shipping, receiving, and tra | Stock clerks and order fille | Data entry keyers
   + skill answers          Secretaries and administrati | Shipping, receiving, and tra | Stock clerks and order fille | Data entry keyers
   + description            Shipping, receiving, and tra | Stock clerks and order fille | Data entry keyers | Receptionists and informatio
   answers only (no title)  Secretaries and administrati | Customer service representat | Childcare workers | Elementary and middle school
#7 Senior Researcher -> Planners, anthropologists, historians & more
   title only               Psychologists | Urban and regional planners | Remote sensing & other physi | Environmental scientists and
   + skill answers          Psychologists | Urban and regional planners | Remote sensing & other physi | Environmental scientists and
   + description            Psychologists | Remote sensing & other physi | Environmental scientists and | M

   title only               IT project managers, QA test | Computer systems analysts | Computer support specialists | Computer programmers
   + skill answers          IT project managers, QA test | Computer programmers | Computer support specialists | Computer systems analysts
   + description            Computer support specialists | Computer programmers | Web developers | Network and computer systems
   answers only (no title)  Assemblers & fabricators | Automotive service technicia | Welding, soldering, and braz | CNC operators & other produc
#9 Postdoctoral researcher -> Biological scientists
   title only               Remote sensing & other physi | Medical scientists | Environmental scientists and | Chemists and materials scien
   + skill answers          Remote sensing & other physi | Medical scientists | Environmental scientists and | Chemists and materials scien
   answers only (no title)  Registered nurses | Nursing, psychiatric, and ho | Physicians and surgeons | Health prac

   title only               Psychologists | Urban and regional planners | Remote sensing & other physi | Environmental scientists and
   + skill answers          Psychologists | Urban and regional planners | Environmental scientists and | Remote sensing & other physi
   + description            Psychologists | Urban and regional planners | Environmental scientists and | Remote sensing & other physi
   answers only (no title)  Clergy | Nursing, psychiatric, and ho | Probation officers and corre | Health educators & community
#11 Social worker -> Social workers
   title only               Counselors | Social and human service ass | Clergy | Health educators & community
   + skill answers          Counselors | Social and human service ass | Clergy | Health educators & community
   + description            Social and human service ass | Clergy | Health educators & community | Probation officers and corre
   answers only (no title)  Security guards and gaming s | Driver/sales workers and tr

   title only               Social workers | Social and human service ass | Clergy | Probation officers and corre
   + skill answers          Social workers | Social and human service ass | Clergy | Probation officers and corre
   + description            Social and human service ass | Clergy | Probation officers and corre | Service sales representative
   answers only (no title)  Registered nurses | Social and human service ass | Physicians and surgeons | Probation officers and corre
#13 Lawyer -> Lawyers
   title only               Physicians and surgeons | Judicial law clerks | Compliance, project & other  | Other teachers and instructo
   + skill answers          Physicians and surgeons | Judicial law clerks | Paralegals and legal assista | Compliance, project & other 
   + description            Physicians and surgeons | Paralegals and legal assista | Compliance, project & other  | Other teachers and instructo
   answers only (no title)  Remote sensing & other physi | Psychologist

   title only               Physicians and surgeons | Judicial law clerks | Compliance, project & other  | Other teachers and instructo
   + skill answers          Physicians and surgeons | Judicial law clerks | Compliance, project & other  | Paralegals and legal assista
   + description            Physicians and surgeons | Paralegals and legal assista | Elementary and middle school | Management analysts
   answers only (no title)  Compliance, project & other  | Accountants and auditors | Chief executives | Management analysts
#15 Social worker -> Social workers
   title only               Counselors | Social and human service ass | Clergy | Health educators & community
   + skill answers          Counselors | Social and human service ass | Clergy | Health educators & community
   + description            Social and human service ass | Clergy | Health educators & community | Probation officers and corre
   answers only (no title)  Clergy | Registered nurses | Health educators & communi

   title only               Psychologists | Urban and regional planners | Remote sensing & other physi | Environmental scientists and
   + skill answers          Psychologists | Urban and regional planners | Environmental scientists and | Medical scientists
   + description            Preschool and kindergarten t | Psychologists | Urban and regional planners | Lawyers
   answers only (no title)  Software developers, applica | Elementary and middle school | CNC operators & other produc | Designers
#17 Web developer -> Web developers
   title only               Software developers, applica | IT project managers, QA test | Computer support specialists | Computer systems analysts
   + skill answers          Software developers, applica | Computer support specialists | IT project managers, QA test | Computer systems analysts
   + description            Software developers, applica | Computer support specialists | Computer systems analysts | Network and computer systems
   answers only (no t


## Findings

TODO (Tim): which outcomes look right, which look wrong, and why.
